# Tool Calling by Hand
**Module 5, Topic 1 — Automation & Agentic Systems**

## The Problem

An LLM can reason about text, but it **cannot** look up a licence in a database, check a live NOC status, or call an API on its own. If you ask "Is Sahel Link Networks compliant?", the model can only guess — it has no access to the case register.

## The Solution

**Tool calling** lets the model request that *you* run a function on its behalf. You describe the function as a JSON Schema, the model decides when to call it and with what arguments, you execute it and return the result, and the model uses the result to finish its answer.

In this notebook you will:
1. Define a tool with a JSON Schema
2. Send a question to the API with the tool definition
3. Execute the function the model requests
4. Return the result and let the model answer
5. Handle multiple tool calls, errors, multiple tools, and build a general loop

Every agent framework — LangGraph, LangChain, CrewAI — is a wrapper over this loop.


## Setup

In [ ]:
import json
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
)
MODEL = os.getenv("MODEL_NAME")

print("Client ready")

## Step 1 — Define a Tool with a JSON Schema

We describe our tool so the model knows what it does, when to use it, and what arguments to pass.

The scenario: a tool that looks up the compliance status of a VSAT operator licence in the case register.

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "lookup_licence_status",
            "description": (
                "Look up the current compliance status of a VSAT operator "
                "licence by its licence number. Returns the operator name, "
                "compliance status, and the reason."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "licence_number": {
                        "type": "string",
                        "description": "The VSAT operator licence number, e.g. VSAT-OP-0192",
                    }
                },
                "required": ["licence_number"],
            },
        },
    }
]

print("Tool definition ready")
print(json.dumps(tools, indent=2))

## Step 2 — Write the Python Function

This is the function the model will ask us to run. It simulates a case register lookup — in production this would query a real database.

In [ ]:
def lookup_licence_status(licence_number: str) -> dict:
    """Simulated case register lookup."""
    register = {
        "VSAT-OP-0192": {
            "operator": "Sahel Link Networks Ltd",
            "status": "non-compliant",
            "reason": "Q4 2025 Quarterly Service Quality Report overdue (due 2026-01-30)",
        },
        "VSAT-OP-0077": {
            "operator": "Delta Coast Broadband Ltd",
            "status": "compliant",
            "reason": "Enforcement case MCE-EC-2025-118 closed 2026-02-02, no outstanding issues",
        },
        "VSAT-OP-0301": {
            "operator": "Harmattan Telecom Ltd",
            "status": "compliant",
            "reason": "All reports submitted, no open cases",
        },
    }
    entry = register.get(licence_number)
    if entry:
        return entry
    return {"error": f"Licence {licence_number} not found in the register"}

# Quick test
print(lookup_licence_status("VSAT-OP-0192"))
print(lookup_licence_status("VSAT-OP-9999"))

## Step 3 — Send the Question with Tool Definitions

We send the user's question and our tool definitions to the API. The model decides whether it needs to call the tool.

In [ ]:
messages = [
    {"role": "user", "content": "Is Sahel Link Networks currently compliant?"}
]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools,
)

message = response.choices[0].message
print("Finish reason:", response.choices[0].finish_reason)
print("Content:", message.content)
print("Tool calls:", message.tool_calls)

## Step 4 — Inspect the Tool Call

The model did not answer with text — it returned a structured request asking us to run a function.

In [ ]:
if message.tool_calls:
    tool_call = message.tool_calls[0]
    print("Function name:", tool_call.function.name)
    print("Arguments (raw string):", tool_call.function.arguments)
    print("Arguments (parsed):", json.loads(tool_call.function.arguments))
    print("Tool call ID:", tool_call.id)
else:
    print("The model answered directly:", message.content)

## Step 5 — Execute the Function

We run the function ourselves with the arguments the model gave us.

In [ ]:
args = json.loads(tool_call.function.arguments)
result = lookup_licence_status(**args)

print("Function result:")
print(json.dumps(result, indent=2))

## Step 6 — Return the Result and Get the Final Answer

We add the assistant's tool-call message and our tool result to the conversation, then let the model generate its answer.

In [ ]:
# Add the assistant's tool-call message
messages.append(message)

# Add the tool result
messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": json.dumps(result),
})

# Second API call — model uses the result to answer
final_response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools,
)

print(final_response.choices[0].message.content)

## Step 7 — Multiple Tool Calls in One Turn

When you ask about two operators at once, the model may return two tool calls in a single response.

In [ ]:
messages_multi = [
    {"role": "user", "content": "Are Sahel Link Networks and Delta Coast Broadband both compliant?"}
]

response_multi = client.chat.completions.create(
    model=MODEL,
    messages=messages_multi,
    tools=tools,
)

msg = response_multi.choices[0].message

if msg.tool_calls:
    print(f"Model requested {len(msg.tool_calls)} tool call(s):")
    for tc in msg.tool_calls:
        print(f"  - {tc.function.name}({tc.function.arguments})")
else:
    print("Model answered directly:", msg.content)

In [ ]:
# Execute all tool calls and return results
messages_multi.append(msg)

for tc in msg.tool_calls:
    args = json.loads(tc.function.arguments)
    result = lookup_licence_status(**args)
    messages_multi.append({
        "role": "tool",
        "tool_call_id": tc.id,
        "content": json.dumps(result),
    })
    print(f"Returned result for {tc.function.name}({args}): {result}")

# Get the combined answer
final_multi = client.chat.completions.create(
    model=MODEL,
    messages=messages_multi,
    tools=tools,
)

print("\nFinal answer:")
print(final_multi.choices[0].message.content)

## Step 8 — Error Handling

What happens when the tool cannot find the licence? Return the error as data — do not let the function crash.

In [ ]:
messages_err = [
    {"role": "user", "content": "What is the compliance status of licence VSAT-OP-9999?"}
]

response_err = client.chat.completions.create(
    model=MODEL,
    messages=messages_err,
    tools=tools,
)

msg_err = response_err.choices[0].message

if msg_err.tool_calls:
    messages_err.append(msg_err)
    for tc in msg_err.tool_calls:
        args = json.loads(tc.function.arguments)
        try:
            result = lookup_licence_status(**args)
        except Exception as e:
            result = {"error": str(e)}
        messages_err.append({
            "role": "tool",
            "tool_call_id": tc.id,
            "content": json.dumps(result),
        })

    final_err = client.chat.completions.create(
        model=MODEL,
        messages=messages_err,
        tools=tools,
    )
    print(final_err.choices[0].message.content)
else:
    print(msg_err.content)

## Step 9 — Multiple Tools — Letting the Model Choose

Add a second tool so the model has a choice. It picks which tool to call based on the descriptions.

In [ ]:
def get_incident_summary(incident_id: str) -> dict:
    """Simulated NOC incident lookup."""
    incidents = {
        "NOC-INC-2026-0417": {
            "incident_id": "NOC-INC-2026-0417",
            "severity": "P2 - High",
            "affected_link": "KU-HUB1-GW3",
            "customers_affected": 3,
            "terminals_affected": 41,
            "duration": "1 hour 38 minutes",
            "root_cause": "Failed cooling fan caused over-temperature shutdown of HPA-2",
            "status": "resolved",
        },
    }
    entry = incidents.get(incident_id)
    if entry:
        return entry
    return {"error": f"Incident {incident_id} not found"}


# Extended tool list
tools_extended = [
    tools[0],  # lookup_licence_status
    {
        "type": "function",
        "function": {
            "name": "get_incident_summary",
            "description": (
                "Get a summary of a NOC incident report by its incident ID. "
                "Returns severity, affected link, customers, duration, and root cause."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "incident_id": {
                        "type": "string",
                        "description": "The NOC incident ID, e.g. NOC-INC-2026-0417",
                    }
                },
                "required": ["incident_id"],
            },
        },
    },
]

# Map function names to callables
available_functions = {
    "lookup_licence_status": lookup_licence_status,
    "get_incident_summary": get_incident_summary,
}

print(f"Tools available: {list(available_functions.keys())}")

In [ ]:
# Helper to run a single question through the tool-calling flow
def ask_with_tools(question, tools_list, functions_map):
    """Send a question, handle one round of tool calls, print the answer."""
    msgs = [{"role": "user", "content": question}]

    resp = client.chat.completions.create(
        model=MODEL, messages=msgs, tools=tools_list,
    )
    msg = resp.choices[0].message

    if not msg.tool_calls:
        print(f"Q: {question}")
        print(f"A (direct): {msg.content}\n")
        return

    print(f"Q: {question}")
    msgs.append(msg)
    for tc in msg.tool_calls:
        fn = functions_map[tc.function.name]
        args = json.loads(tc.function.arguments)
        print(f"   Tool called: {tc.function.name}({args})")
        try:
            result = fn(**args)
        except Exception as e:
            result = {"error": str(e)}
        msgs.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps(result)})

    final = client.chat.completions.create(
        model=MODEL, messages=msgs, tools=tools_list,
    )
    print(f"A: {final.choices[0].message.content}\n")


# Test: compliance question -> licence tool
ask_with_tools("Is Sahel Link Networks compliant?", tools_extended, available_functions)

# Test: incident question -> incident tool
ask_with_tools("What happened with NOC incident 0417?", tools_extended, available_functions)

# Test: general question -> no tool needed
ask_with_tools("What is the capital of Nigeria?", tools_extended, available_functions)

## Step 10 — The General Tool-Calling Loop

A reusable function that handles any number of tools and any number of rounds. The `max_rounds` parameter prevents infinite loops.

In [ ]:
def tool_loop(question, tools_list, functions_map, max_rounds=5, verbose=True):
    """Run the tool-calling loop until the model gives a text answer."""
    messages = [{"role": "user", "content": question}]

    for round_num in range(1, max_rounds + 1):
        response = client.chat.completions.create(
            model=MODEL, messages=messages, tools=tools_list,
        )
        message = response.choices[0].message

        if not message.tool_calls:
            if verbose:
                print(f"[Round {round_num}] Model answered directly.")
            return message.content

        if verbose:
            print(f"[Round {round_num}] {len(message.tool_calls)} tool call(s):")

        messages.append(message)
        for tc in message.tool_calls:
            fn = functions_map.get(tc.function.name)
            args = json.loads(tc.function.arguments)
            if verbose:
                print(f"  -> {tc.function.name}({args})")
            if fn is None:
                result = {"error": f"Unknown function: {tc.function.name}"}
            else:
                try:
                    result = fn(**args)
                except Exception as e:
                    result = {"error": str(e)}
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": json.dumps(result),
            })

    return "Reached maximum tool-calling rounds."


print("tool_loop() defined")

In [ ]:
# Test the general loop
print("--- Question 1 ---")
answer = tool_loop(
    "Is Sahel Link Networks compliant, and what happened with NOC incident 0417?",
    tools_extended,
    available_functions,
)
print(f"\nAnswer: {answer}")

In [ ]:
print("--- Question 2 (no tool needed) ---")
answer = tool_loop(
    "Explain what tool calling means in one sentence.",
    tools_extended,
    available_functions,
)
print(f"\nAnswer: {answer}")

In [ ]:
print("--- Question 3 (error case) ---")
answer = tool_loop(
    "What is the compliance status of licence VSAT-OP-5555?",
    tools_extended,
    available_functions,
)
print(f"\nAnswer: {answer}")

## Summary

You built the entire tool-calling pattern from scratch:

1. **Define** a tool with a JSON Schema (name, description, parameters)
2. **Send** the question and tool definitions to the API
3. **Execute** the function the model requested
4. **Return** the result as a `role="tool"` message with the matching `tool_call_id`
5. **Repeat** until the model gives a text answer

Every agent framework — LangGraph, LangChain, CrewAI — is a wrapper over this loop.

**Next:** Topic 2 — LangGraph: Nodes, Edges & State

## Exercises

### Exercise 1 — Add a third tool
Define a `get_operator_contact` tool that takes a `licence_number` and returns a dict with `operator`, `email`, and `phone` (simulated data). Add it to `tools_extended` and `available_functions`, then ask: *"How do I reach the operator of VSAT-OP-0301?"* Confirm the model picks the new tool.

### Exercise 2 — Force tool use with `tool_choice`
Pass `tool_choice={"type": "function", "function": {"name": "lookup_licence_status"}}` to `client.chat.completions.create` for the question *"What is the capital of Nigeria?"* — a question that normally needs no tool. Observe what happens when the model is forced to call a tool it doesn't need.

### Exercise 3 — Add round counting to `tool_loop`
Modify `tool_loop` to return both the answer and the number of rounds it took. Test with a question that needs two tools (compliance + incident) and verify it completes in one or two rounds.

**Time:** 20 minutes

In [ ]:
# Exercise 1 — your code here


In [ ]:
# Exercise 2 — your code here


In [ ]:
# Exercise 3 — your code here
